# Data types: get every column's dtype right_Data Wrangling_**Numbers stuck as text, booleans as "Yes"/"No", dates as strings — fix the dtypes before you do anything else.**A pandas DataFrame is a stack of typed columns. Each column carries a dtype &mdash; one label       that says what the values are and, crucially, how operations behave on them. The dtype is not       cosmetic: it decides whether + means "add" or "glue text together", whether       < compares magnitudes or alphabetical order, and how many bytes each value costs.       Raw data almost never arrives with the right dtypes. A CSV has no types at all; an API mixes them;       a database stores flags as words. So the first wrangling job is to walk every column and force       its dtype to match its meaning. Get this right and math, sorting, grouping, joining, and plotting all       "just work". Get it wrong and they fail silently &mdash; no error, just wrong answers.---This notebook fixes a messy frame's dtypes one column-type at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandasWe'll take a frame where every column arrived as text — exactly how a CSV or API hands it over — and fix one dtype problem at a time. We do it in four steps: (1) see the mess, (2) parse numbers, booleans, and dates, (3) pick the right integer and category dtypes, and (4) confirm that math, grouping, and date logic now work.

### Step 1 — Build the messy frame and inspect dtypesHere is the frame as raw data would deliver it: numbers wrapped in `$`/`%`, booleans spelled `Yes`/`No`, dates as strings, and a low-cardinality `plan` column. Printing `df.dtypes` shows every column is `object` — so `+` glues text, `<` sorts alphabetically, and any math is silently wrong.

In [ ]:
import numpy as np
import pandas as pd

# A messy frame, exactly as a CSV / API would hand it to you.
df = pd.DataFrame({
    'customer_id': ['1001', '1002', '1003', None, '1005'],   # whole-number IDs, one missing
    'revenue':     ['$1,234', '$5', '$78', '$2,400', '$960'], # currency strings
    'growth':      ['12%', '5%', '120%', '0%', '8%'],         # percent strings
    'is_paid':     ['Yes', 'No', 'Yes', 'Yes', 'No'],         # boolean as words
    'signup':      ['2026-01-05','2026-03-20','2026-06-01',   # dates as strings
                    '2026-02-14','2026-05-30'],
    'plan':        ['free','pro','free','enterprise','pro'],  # small fixed set -> category
})

print(df.dtypes)        # EVERYTHING is 'object' -> math/sort/group are all broken
# customer_id    object
# revenue        object   ... etc.

### Step 2 — Parse numbers, booleans, and datesNumbers stuck as strings need their junk characters stripped before `pd.to_numeric` can coerce them — `errors='coerce'` turns anything unparseable into `NaN`, so we assert the coercion left no gaps. Booleans map cleanly from `Yes`/`No`, and date strings become real `datetime64` (passing an explicit `format` is faster and removes ambiguity).

In [ ]:
# Numbers stuck as strings: strip junk chars, then coerce to numeric.
df['revenue'] = pd.to_numeric(
    df['revenue'].str.replace('$', '', regex=False)
                 .str.replace(',', '', regex=False),
    errors='coerce')                                   # -> int/float, NaN if unparseable
assert df['revenue'].isna().sum() == 0                 # always check the coercion!

df['growth'] = pd.to_numeric(
    df['growth'].str.replace('%', '', regex=False),
    errors='coerce') / 100.0                            # 12% -> 0.12

# Boolean stored as 'Yes'/'No'.
df['is_paid'] = df['is_paid'].map({'Yes': True, 'No': False}).astype(bool)

# Date strings -> datetime64 (pass format for speed + no ambiguity).
df['signup'] = pd.to_datetime(df['signup'], format='%Y-%m-%d')

### Step 3 — Pick the right integer and category dtypesA plain `int64` can't hold a missing value, so it would upcast the IDs to `1001.0`; the nullable `Int64` keeps them whole while still allowing the gap. And a low-cardinality text column like `plan` becomes far cheaper as a `category` — pandas stores each label once plus a small code per row, which also speeds up groupby.

In [ ]:
# Nullable integer: keep IDs whole AND allow the missing row.
df['customer_id'] = pd.to_numeric(df['customer_id']).astype('Int64')  # 1001, not 1001.0
# (plain int64 would upcast to float the moment a value is missing)

# Low-cardinality text -> category (memory + faster groupby).
before = df['plan'].memory_usage(deep=True)
df['plan'] = df['plan'].astype('category')
after = df['plan'].memory_usage(deep=True)
print('plan memory  before:', before, ' after:', after)   # after is much smaller

### Step 4 — Confirm everything now worksWith each column carrying its true dtype, the operations that were broken in Step 1 behave correctly: `.sum()` adds instead of concatenating, the boolean mean is the fraction paid, groupby works on the category, and `.dt` unlocks date math.

In [ ]:
print(df.dtypes)
# customer_id            Int64
# revenue               int64
# growth              float64
# is_paid                bool
# signup       datetime64[ns]
# plan               category

# Now everything WORKS:
print(df['revenue'].sum())              # 4677 (adds, not concatenates)
print(df['is_paid'].mean())             # 0.6  (fraction paid)
print(df.groupby('plan')['revenue'].mean())   # group by the category
print(df['signup'].dt.month.tolist())   # date math unlocked

## Visualize the data & results_Take a 5,000-row frame of repeated text labels and currency strings. How much memory does each column use as an object dtype, versus after converting the labels to 'category' and the amount to a downcast int?_

### Step 1 — Build a 5,000-row messy frameWe generate a realistic frame: several repeated text labels (`city`, `plan`, `status`), a boolean stored as words, and currency strings. Every one of these columns lands as the heavy `object` dtype, which is what we'll measure and then shrink.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(0)
n = 5000
cities = ['San Francisco','New York','Austin','Seattle','Boston','Chicago']
plans = ['free','basic','pro','enterprise']
status = ['active','churned','trial']

df = pd.DataFrame({
    'city':    rng.choice(cities, n),
    'plan':    rng.choice(plans, n),
    'status':  rng.choice(status, n),
    'is_paid': rng.choice(['Yes','No'], n),                       # bool stored as words
    'revenue': ['$' + str(int(v)) for v in rng.gamma(2, 50, n)],  # currency strings
})

### Step 2 — Measure the object-dtype memory`memory_usage(deep=True)` measures the *true* cost of object columns (it follows the pointers to the actual Python strings). Each of these columns eats hundreds of kilobytes because every row stores its own string.

In [ ]:
cols = ['city','plan','status','is_paid','revenue']
before = df.memory_usage(deep=True)
print('BEFORE (KB):', {c: round(before[c]/1024, 1) for c in cols})
# -> {'city': 316.7, 'plan': 305.3, 'status': 307.6, 'is_paid': 290.6, 'revenue': 294.9}

### Step 3 — Convert dtypes and compareConverting the four repeated-label columns to `category` and parsing `revenue` into a downcast `int32` collapses the memory dramatically — each category stores its labels once plus a tiny per-row code. The before/after totals show roughly a 37x shrink.

In [ ]:
after_df = df.copy()
for c in ['city','plan','status','is_paid']:
    after_df[c] = after_df[c].astype('category')            # text -> category
after_df['revenue'] = pd.to_numeric(
    after_df['revenue'].str.replace('$','',regex=False)).astype('int32')

after = after_df.memory_usage(deep=True)
print('AFTER  (KB):', {c: round(after[c]/1024, 1) for c in cols})
# -> {'city': 5.4, 'plan': 5.3, 'status': 5.2, 'is_paid': 5.1, 'revenue': 19.5}
print('total before/after (KB):',
      round(before[cols].sum()/1024,1), '/', round(after[cols].sum()/1024,1))
# -> 1515.0 / 40.5   (~37x smaller)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A column price reads ["$1,200", "$950", "$3,499"]. You call df['price'].sum() and get "$1,200$950$3,499", and df.sort_values('price') puts the $1,200 row above the $950 row. Diagnose and fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the dtype with df['price'].dtype &mdash; it is object (strings). — _For an object column, + concatenates and < compares text lexicographically, which is exactly what you observed._
- Strip the junk: s = df['price'].str.replace('$','',regex=False).str.replace(',','',regex=False). — _pd.to_numeric only parses bare digits; the $ and , must go first._
- Parse to numbers: df['price'] = pd.to_numeric(s, errors='coerce'), then check df['price'].isna().sum() is 0. — _Now the dtype is numeric, so .sum() adds and sorting compares magnitude. The NaN check confirms nothing failed to parse silently._

**Answer:** The column is object dtype (strings), so + concatenates and sorting is lexicographic ("$1,200" < "$950" because "1" < "9"). Fix: str.replace to drop "$" and ",", then pd.to_numeric(..., errors='coerce'), then confirm .isna().sum() is 0. After that .sum() gives 5649 and the sort is correct.

</details>

**Problem 2.** You load a 2-million-row table. One column region holds 6 distinct text labels repeated millions of times and df.info() shows it eating hundreds of megabytes. A customer_id column is whole numbers but a few rows are missing, and pandas shows it as float64 with values like 10042.0. Fix both.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Convert region with df['region'] = df['region'].astype('category'). — _With only 6 distinct labels over millions of rows, category stores each label once plus a 1-byte code per row &mdash; a huge memory drop and faster groupby._
- Convert customer_id with df['customer_id'] = df['customer_id'].astype('Int64'). — _The nullable Int64 dtype keeps the IDs as whole numbers while still allowing the missing rows, so 10042.0 goes back to 10042 and <NA> marks the gaps._
- Re-run df.info(memory_usage='deep') to confirm the shrink and the new dtypes. — _deep measures the true object-string cost so you can see the before/after honestly._

**Answer:** region is a high-memory object column: convert it to category (astype('category')) &mdash; few labels, many rows, so memory collapses and grouping speeds up. customer_id was upcast to float64 because plain int64 can't hold the missing rows; convert it to the nullable Int64 so the IDs stay whole and the gaps become <NA>.

</details>

**Problem 3.** A survey column satisfaction holds "low", "medium", "high". You convert it with astype('category'), then try df[df['satisfaction'] > 'low'] and it errors or behaves alphabetically. What did you miss?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that a plain category is unordered by default. — _Without an order, > has no defined meaning, and if compared as text it would order "high" < "low" < "medium" alphabetically &mdash; wrong._
- Build an ordered dtype: from pandas.api.types import CategoricalDtype; cat = CategoricalDtype(['low','medium','high'], ordered=True). — _This pins the real ranking low < medium < high onto the category._
- Apply it: df['satisfaction'] = df['satisfaction'].astype(cat), then df[df['satisfaction'] > 'low'] works. — _Now comparisons and sorting follow the meaningful order, not the alphabet._

**Answer:** You missed the order. A plain category is unordered, so > is undefined (or falls back to alphabetical, which mis-ranks "high"). Build an ordered category with CategoricalDtype(['low','medium','high'], ordered=True) and apply it; then low < medium < high holds and comparisons/sorting are correct.

</details>